## PyTorch Exploration and Telco Customer Churn Prediction

#Objective

The objective of this notebook is to explore the fundamentals of PyTorch , understand tensors an and build a simple neural network for predicting customer churn using our already cleaned and featured Telco Customer Churn dataset

# What is PyTorch?

PyTorch is an open-source deep learning framework developed by Meta AI. It provides an easy way to build neural networks, perform tensor computations, and automatically calculate gradients using Autograd.

PyTorch is widely used in machine learning research, computer vision, natural language processing, and artificial intelligence applications.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from torch.utils.data import TensorDataset, DataLoader

In [2]:
print("PyTorch Version:", torch.__version__)

PyTorch Version: 2.13.0+cpu


In [3]:
print(torch.device("cpu"))
print(torch.cuda.is_available())

cpu
False


In [4]:
scalar = torch.tensor(5)

vector = torch.tensor([1,2,3,4])

matrix = torch.tensor([[1,2],[3,4]])

random_tensor = torch.randn(4,2)

In [5]:
print(random_tensor)

print(random_tensor.dtype)

print(random_tensor.device)

print(random_tensor.ndim)

print(random_tensor.shape)

print(random_tensor.size())

tensor([[ 0.0059, -1.4220],
        [ 0.3745, -0.2633],
        [-1.7183,  1.0085],
        [ 2.0140, -0.5421]])
torch.float32
cpu
2
torch.Size([4, 2])
torch.Size([4, 2])


In [6]:
A=torch.randn(4,2)

B=torch.randn(2,3)

C=A@B

print(C)
print(C.shape)

tensor([[-0.3062,  0.1885, -0.5855],
        [ 0.5425, -0.0118,  1.0742],
        [-0.1460, -0.7039, -0.3701],
        [ 0.0983,  0.2146,  0.2194]])
torch.Size([4, 3])


In [7]:
w=torch.tensor([2.0],requires_grad=True)

x=torch.tensor([3.0])

y=w*x

loss=y**2

loss.backward()

print(loss)

print(w.grad)

tensor([36.], grad_fn=<PowBackward0>)
tensor([36.])


## Results and Explanation

- *PyTorch Version* : `2.13.0+cpu` means PyTorch is installed and running on the **CPU**. `False` confirms that CUDA (GPU support) is not available.

- *Tensor Properties :* The random tensor has data type `float32`, It is stored on the CPU and has 2 dimensions and a shape / size of (4, 2) => (4 rows and 2 columns).

- *Tensor Operations:* Multiplying a (4 × 2) tensor with a (2 × 3) tensor produces a new tensor of shape (4 × 3) .

- *Autograd :* The output `tensor([36.])` is the loss value and `w.grad = tensor([36.])` is the gradient automatically calculated by PyTorch using backpropagation. This shows how PyTorch computes gradients without manually writing the derivative.

In [8]:
df=pd.read_csv("../data/fully_featured_telco_customer_churn.csv")

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServices,IsNewCustomer,AverageMonthlySpend,HasAutoPayment,IsLongTermCustomer,HighMonthlyCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Electronic check,29.85,29.85,No,1,1,29.850000,0,0,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,Mailed check,56.95,1889.50,No,3,0,55.573529,0,1,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Mailed check,53.85,108.15,Yes,3,1,54.075000,0,0,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Bank transfer (automatic),42.30,1840.75,No,3,0,40.905556,1,1,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Electronic check,70.70,151.65,Yes,1,1,75.825000,0,0,1


In [9]:
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No":0, "Yes":1})

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

C:\Users\HADIN\AppData\Local\Temp\ipykernel_10008\2971937678.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [13]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [14]:
train_dataset = TensorDataset(X_train, y_train)

test_dataset = TensorDataset(X_test, y_test)

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [16]:
class ChurnNet(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [17]:
input_size = X_train.shape[1]
model = ChurnNet(input_size)
print(model)

ChurnNet(
  (network): Sequential(
    (0): Linear(in_features=51, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [18]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [23]:
epochs = 30

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        average_loss = running_loss / len(train_loader)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {average_loss:.4f}")

Epoch [5/30], Loss: 0.3544
Epoch [10/30], Loss: 0.3463
Epoch [15/30], Loss: 0.3387
Epoch [20/30], Loss: 0.3259
Epoch [25/30], Loss: 0.3161
Epoch [30/30], Loss: 0.3062


In [24]:
model.eval()

with torch.no_grad():

    outputs = model(X_test)
    probabilities = torch.sigmoid(outputs)
    predictions = (probabilities >= 0.5).float()

In [25]:
y_true = y_test.numpy()
y_pred = predictions.numpy()
y_prob = probabilities.numpy()

In [26]:
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))
print("ROC-AUC :", roc_auc_score(y_true, y_prob))
print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report")
print(classification_report(y_true, y_pred))

Accuracy : 0.7792760823278921
Precision: 0.592375366568915
Recall : 0.5401069518716578
F1 Score : 0.5650349650349651
ROC-AUC : 0.8110568601617194

Confusion Matrix
[[896 139]
 [172 202]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.84      0.87      0.85      1035
         1.0       0.59      0.54      0.57       374

    accuracy                           0.78      1409
   macro avg       0.72      0.70      0.71      1409
weighted avg       0.77      0.78      0.78      1409



# Results

The trained neural network successfully completed the learning process and produced meaningful predictions for the Telco Customer Churn dataset.

## Final Performance

| Metric    |      Value |
| --------- | ---------: |
| Accuracy  | **77.93%** |
| Precision | **59.24%** |
| Recall    | **54.01%** |
| F1-Score  | **56.50%** |
| ROC-AUC   | **0.8111** |

## Confusion Matrix

|            | Predicted No | Predicted Yes |
| ---------- | -----------: | ------------: |
| Actual No  |          896 |           139 |
| Actual Yes |          172 |           202 |

## Classification Summary

* The model correctly classified the majority of customers.
* Customer churn was detected with moderate recall and precision.
* The neural network demonstrated stable learning throughout training.
* Training loss consistently decreased across all epochs, indicating successful optimization.


# Observations

* Training loss steadily decreased throughout the training process, indicating that the neural network successfully learned from the data.
* Increasing the number of training epochs reduced the training loss further but did not consistently improve evaluation metrics, suggesting the onset of overfitting.
* Logistic Regression achieved better overall Accuracy, Precision, F1-Score, and ROC-AUC on this structured tabular dataset.
* The neural network achieved a slightly higher Recall, demonstrating its ability to identify more customers likely to churn.
* This experiment showed that neural networks are not always the best choice for small tabular datasets, where classical machine learning models often remain highly competitive.

---

# Learning Outcomes

By completing this project, the following concepts were learned:

* Understanding the role of PyTorch in deep learning.
* Creating and manipulating tensors.
* Exploring tensor properties such as dtype, device, shape, size, and dimensions.
* Performing tensor operations and matrix multiplication.
* Understanding Automatic Differentiation (Autograd).
* Building neural networks using `nn.Module`.
* Creating datasets using `TensorDataset`.
* Loading mini-batches using `DataLoader`.
* Defining loss functions using `BCEWithLogitsLoss`.
* Optimizing model parameters using the Adam optimizer.
* Implementing the complete training loop consisting of forward propagation, loss computation, backpropagation, and weight updates.
* Evaluating deep learning models using standard classification metrics.
* Comparing neural networks with classical machine learning algorithms and interpreting their performance.
